## Question 1 

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.stats import bootstrap

DATA_DIR = Path("data/AReM")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

assert DATA_DIR.exists(), "Check your data folder path."

files = list(DATA_DIR.glob("*/dataset*.csv"))
print("Number of instances:", len(files))

Number of instances: 88


In [2]:
def load_instance(path):
    rows = []

    for line in path.read_text().splitlines():
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        # Some files use spaces instead of commas.
        values = re.split(r"[,\s]+", line.rstrip(","))
        rows.append([float(x) for x in values])

    columns = ["time", "ts1", "ts2", "ts3", "ts4", "ts5", "ts6"]
    df = pd.DataFrame(rows, columns=columns)

    if len(df) != 480:
        print(f"Note: {path.parent.name}/{path.name} has {len(df)} rows.")

    assert not df.empty, f"Empty file: {path.name}"
    assert not df.isna().any().any(), f"Missing data: {path.name}"

    return df

In [3]:
instances = []

for folder in sorted(DATA_DIR.iterdir()):
    if not folder.is_dir():
        continue

    for file in sorted(folder.glob("dataset*.csv")):
        dataset_id = int(re.search(r"dataset(\d+)", file.stem).group(1))

        instances.append({
            "activity": folder.name,
            "dataset_id": dataset_id,
            "data": load_instance(file)
        })

print("Total instances:", len(instances))
assert len(instances) == 88

print("Activity:", instances[0]["activity"])
display(instances[0]["data"].head())

Note: sitting/dataset8.csv has 479 rows.
Total instances: 88
Activity: bending1


,time,ts1,ts2,ts3,ts4,ts5,ts6
0,0.0,39.25,0.43,22.75,0.43,33.75,1.3
1,250.0,39.25,0.43,23.00,0.00,33.00,0.0
2,500.0,39.25,0.43,23.25,0.43,33.00,0.0
3,750.0,39.50,0.50,23.00,0.71,33.00,0.0
4,1000.0,39.50,0.50,24.00,0.00,33.00,0.0


In [4]:
for instance in instances:
    activity = instance["activity"]
    dataset_id = instance["dataset_id"]

    if activity in ["bending1", "bending2"]:
        test_ids = [1, 2]
    else:
        test_ids = [1, 2, 3]

    instance["split"] = "test" if dataset_id in test_ids else "train"

metadata = pd.DataFrame([
    {
        "activity": x["activity"],
        "dataset_id": x["dataset_id"],
        "split": x["split"]
    }
    for x in instances
])

display(pd.crosstab(metadata["activity"], metadata["split"]))

print("Training instances:", (metadata["split"] == "train").sum())
print("Test instances:", (metadata["split"] == "test").sum())

split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


Training instances: 69
Test instances: 19


### 1(c)i — Time-domain features

Some common time-domain features are the minimum, maximum, mean,
median, standard deviation, and quartiles. The mean and median
describe the typical values in a signal, while the standard deviation
shows how much the values vary. The range and interquartile range
also describe the spread. Other features include skewness and
kurtosis, which describe the shape of the distribution.

Reference:
tsfresh documentation, “Feature calculators.”
https://tsfresh.readthedocs.io/en/stable/_modules/tsfresh/feature_extraction/feature_calculators.html

In [5]:
def extract_features(df):
    features = {}

    for i in range(1, 7):
        values = df[f"ts{i}"]

        features[f"min{i}"] = values.min()
        features[f"max{i}"] = values.max()
        features[f"mean{i}"] = values.mean()
        features[f"median{i}"] = values.median()
        features[f"std{i}"] = values.std(ddof=1)
        features[f"q1_{i}"] = values.quantile(0.25)
        features[f"q3_{i}"] = values.quantile(0.75)

    return features

In [6]:
records = []

for instance in instances:
    row = {
        "activity": instance["activity"],
        "dataset_id": instance["dataset_id"],
        "split": instance["split"]
    }

    row.update(extract_features(instance["data"]))
    records.append(row)

feature_table = pd.DataFrame(records)

feature_cols = [
    col for col in feature_table.columns
    if col not in ["activity", "dataset_id", "split"]
]

print("Feature matrix shape:", feature_table[feature_cols].shape)
display(feature_table.head())

feature_table.to_csv(OUTPUT_DIR / "features.csv", index=False)

Feature matrix shape: (88, 42)


,activity,dataset_id,split,min1,max1,mean1,median1,std1,q1_1,q3_1,...,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
0,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


In [7]:
def sample_std(values, axis=-1):
    return np.std(values, axis=axis, ddof=1)


results = []
rng = np.random.default_rng(552)

for col in feature_cols:
    values = feature_table[col].to_numpy()

    result = bootstrap(
        (values,),
        statistic=sample_std,
        vectorized=True,
        confidence_level=0.90,
        n_resamples=10000,
        method="percentile",
        random_state=rng
    )

    results.append({
        "feature": col,
        "std": sample_std(values),
        "lower": result.confidence_interval.low,
        "upper": result.confidence_interval.high
    })

ci_table = pd.DataFrame(results)

display(ci_table.round(4))

ci_table.to_csv(
    OUTPUT_DIR / "feature_std_confidence_intervals.csv",
    index=False
)

,feature,std,lower,upper
0,min1,9.5700,8.2569,10.7417
1,max1,4.3944,3.3280,5.3082
2,mean1,5.3357,4.7128,5.8836
3,median1,5.4401,4.8092,5.9993
4,std1,1.7722,1.5726,1.9494
5,q1_1,6.1536,5.5706,6.6526
6,q3_1,5.1389,4.3307,5.8409
7,min2,0.0000,0.0000,0.0000
8,max2,5.0627,4.6327,5.4113
9,mean2,1.5742,1.3908,1.7068


### 1(c)iii — Standard deviations and confidence intervals

I calculated the sample standard deviation of each feature across
the 88 instances. I then used 10,000 bootstrap samples to estimate
a 90% confidence interval, taking the 5th and 95th percentiles of
the bootstrap standard deviations as the lower and upper bounds.


In [8]:
selected_cols = [
    f"{stat}{i}"
    for stat in ["mean", "std", "max"]
    for i in range(1, 7)
]

train_features = feature_table[
    feature_table["split"] == "train"
]

summary = train_features.groupby("activity")[selected_cols].mean()
display(summary.round(2))

,mean1,mean2,mean3,mean4,mean5,mean6,std1,std2,std3,std4,std5,std6,max1,max2,max3,max4,max5,max6
activity,,,,,,,,,,,,,,,,,,
bending1,43.05,0.49,18.99,0.77,28.82,0.55,2.10,0.39,3.70,0.84,2.81,0.51,47.38,2.12,28.27,6.24,35.78,2.80
bending2,31.12,0.41,13.00,0.75,17.81,1.02,4.98,0.59,5.44,0.90,5.12,0.93,44.00,5.92,25.96,7.14,33.33,7.92
cycling,36.62,2.79,16.37,2.99,16.98,3.07,3.84,1.93,3.45,1.73,3.02,1.73,44.40,10.84,26.70,9.35,24.44,9.18
lying,39.82,0.39,7.61,0.57,9.88,0.65,1.78,0.29,3.14,0.60,3.61,0.56,42.87,2.61,14.83,4.10,18.18,4.14
sitting,41.06,0.42,14.68,0.73,15.86,0.90,2.27,0.38,4.72,0.83,4.59,0.80,45.82,3.24,23.56,5.72,24.94,5.83
standing,43.97,0.38,15.18,0.59,13.52,0.92,2.08,0.37,3.11,0.69,3.23,0.72,47.13,2.85,21.61,5.46,20.88,5.27
walking,34.51,4.37,15.37,3.19,15.91,3.34,4.73,2.46,2.90,1.61,3.09,1.69,46.56,14.08,23.42,9.11,23.94,9.10


### 1(c)iv — Feature selection

I chose mean, standard deviation, and maximum.

The mean helps distinguish activities with different signal levels.
For example, the average mean1 is 43.05 for bending1 and 31.12
for bending2.

Standard deviation captures how much the signal changes within
each recording. The average std2 is 2.46 for walking, compared
with 0.29 for lying, suggesting that it may help distinguish
these activities.

The maximum captures peak values. For example, the average max2
is 14.08 for walking and 2.61 for lying.

These differences make the three feature types feasible choices,
although group averages alone do not establish which features
will perform best in classification.


## Question 2 — ISLR 3.7.4

### (a)
The cubic model will have a training RSS that is lower than or
equal to the linear model's. Setting β₂ and β₃ to zero gives the
linear model, so the cubic model can always fit the training
data at least as well.

### (b)
I would expect the linear model to have a lower test RSS because
the true relationship is linear. The extra terms in the cubic
model may fit noise rather than the actual relationship. This
is an expectation, though, not a guarantee for every test set.

### (c)
The cubic model will again have a training RSS that is lower
than or equal to the linear model's. Adding more terms cannot
increase the training RSS when both models are fitted by least
squares, even if the true relationship is nonlinear.

### (d)
There is not enough information to tell. The cubic model may
do better if its extra terms capture the nonlinear relationship.
But if the relationship is nearly linear, those terms may mostly
fit noise and lead to a higher test RSS.